In [1]:
# Connect Google Drive (Colab)
from google.colab import drive

drive.mount('/content/drive')
print('Connected to the drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Connected to the drive


# STEP 6: Dataset Explorer & Validation

Run this after steps 01-04 to:
- Understand the shape and quality of your compiled dataset
- Check label distributions
- Spot missing data
- Preview sample rows per label class

Run: python 06_explore_dataset.py

In [2]:
import pandas as pd
from pathlib import Path
import pyarrow.parquet as pq

BASE_DIR = Path('/content/drive/MyDrive/MiniProject')

# Step 1 handles raw .dta ingestion; Step 6 explores processed parquet outputs.
COMPILED_DIR_PRIMARY = BASE_DIR / 'compiled_dataset_180426'
COMPILED_DIR_FALLBACK = BASE_DIR / 'compiled_dataset'
IHCJ_DIR = BASE_DIR / 'IHCJ_dataset_180426'
ISCJ_DIR = BASE_DIR / 'ISCJ_dataset_180426'

# Tuned for ~10 GB RAM Colab runtime.
MAX_ROWS_PER_FILE = 75_000
PARQUET_BATCH_SIZE = 25_000
SAMPLE_PER_LABEL = 3

LABEL_MAP = {
    -1: "Unknown (needs labeling)",
    0:  "NOT ADR/ODR eligible",
    1:  "ADR eligible only",
    2:  "ADR + ODR eligible",
}


def first_existing(candidates):
    for p in candidates:
        if p.exists():
            return p
    return None


def first_matching(glob_dirs, pattern):
    for d in glob_dirs:
        if d.exists():
            matches = sorted(d.glob(pattern))
            if matches:
                return matches[0]
    return None


def read_parquet_head(path: Path, max_rows: int = MAX_ROWS_PER_FILE, columns=None):
    """Read first N rows from parquet in streaming batches (low RAM)."""
    pf = pq.ParquetFile(path)
    total_rows = pf.metadata.num_rows if pf.metadata is not None else None

    available_cols = pf.schema.names
    if columns is None:
        use_cols = available_cols
    else:
        use_cols = [c for c in columns if c in available_cols]

    if not use_cols:
        return pd.DataFrame(), total_rows, False

    chunks = []
    loaded = 0

    for batch in pf.iter_batches(batch_size=PARQUET_BATCH_SIZE, columns=use_cols):
        chunk = batch.to_pandas()
        if chunk.empty:
            continue

        remaining = max_rows - loaded
        if remaining <= 0:
            break

        if len(chunk) > remaining:
            chunk = chunk.iloc[:remaining].copy()

        chunks.append(chunk)
        loaded += len(chunk)

        if loaded >= max_rows:
            break

    if not chunks:
        return pd.DataFrame(columns=use_cols), total_rows, False

    df = pd.concat(chunks, ignore_index=True)
    was_truncated = (total_rows is not None and total_rows > len(df))
    return df, total_rows, was_truncated


print(f'Compiled dirs checked: {COMPILED_DIR_PRIMARY} | {COMPILED_DIR_FALLBACK}')
print(f'HC dir checked: {IHCJ_DIR}')
print(f'SC dir checked: {ISCJ_DIR}')
print(f'Low-RAM mode: MAX_ROWS_PER_FILE={MAX_ROWS_PER_FILE:,}, PARQUET_BATCH_SIZE={PARQUET_BATCH_SIZE:,}')

Compiled dirs checked: /content/drive/MyDrive/MiniProject/compiled_dataset_180426 | /content/drive/MyDrive/MiniProject/compiled_dataset
HC dir checked: /content/drive/MyDrive/MiniProject/IHCJ_dataset_180426
SC dir checked: /content/drive/MyDrive/MiniProject/ISCJ_dataset_180426
Low-RAM mode: MAX_ROWS_PER_FILE=75,000, PARQUET_BATCH_SIZE=25,000


In [3]:
def show_section(title: str):
    print(f"\n{'═' * 60}")
    print(f"  {title}")
    print('═' * 60)

In [4]:
def _read_for_exploration(path: Path):
    """Read parquet safely in batches and cap rows to avoid kernel OOM."""
    if path.suffix.lower() != '.parquet':
        raise ValueError(f'Expected parquet file, got: {path.suffix}')

    df, total_rows, was_truncated = read_parquet_head(path, max_rows=MAX_ROWS_PER_FILE)
    return df, was_truncated, total_rows


def explore(path: Path, name: str):
    if not path.exists():
        print(f"  [SKIP] {path} not found.")
        return

    try:
        df, was_sampled, total_rows = _read_for_exploration(path)
    except Exception as e:
        print(f"  [ERROR] Failed to read {path.name}: {e}")
        return

    if df.empty:
        print(f"  [SKIP] {path} has no readable rows.")
        return

    sampled_note = ""
    if was_sampled:
        if total_rows is not None:
            sampled_note = f" (sampled first {len(df):,} of {total_rows:,} rows)"
        else:
            sampled_note = f" (sampled first {len(df):,} rows)"

    show_section(f"{name}  ({len(df):,} rows{sampled_note})")

    print(f"\nColumns ({len(df.columns)}):")
    print("  " + ", ".join(df.columns.tolist()))

    print("\nData types:")
    print(df.dtypes.to_string())

    print("\nNull counts (top columns):")
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0].sort_values(ascending=False)
    if len(nulls):
        print(nulls.head(15).to_string())
    else:
        print("  No nulls found.")

    if "final_label" in df.columns:
        print("\nLabel distribution:")
        counts = df["final_label"].map(LABEL_MAP).value_counts(dropna=False)
        total = len(df)
        for label, count in counts.items():
            print(f"  {str(label):<35} {count:>8,}  ({count/total*100:.1f}%)")

    if "adr_target" in df.columns:
        print("\nADR target distribution:")
        print(df["adr_target"].value_counts(dropna=False).to_string())

    if "odr_target" in df.columns:
        print("\nODR target distribution:")
        print(df["odr_target"].value_counts(dropna=False).to_string())

    if "source" in df.columns:
        print("\nSource distribution:")
        print(df["source"].value_counts(dropna=False).to_string())

    if "court_level" in df.columns:
        print("\nCourt level:")
        print(df["court_level"].value_counts(dropna=False).to_string())

    if "year" in df.columns:
        print(f"\nYear range: {df['year'].min()} - {df['year'].max()}")
        print(df["year"].value_counts(dropna=False).sort_index().to_string())

    if "act" in df.columns:
        print("\nTop 20 Acts:")
        print(df["act"].value_counts(dropna=False).head(20).to_string())

    if "final_label" in df.columns:
        for label_val in [0, 1, 2]:
            subset = df[df["final_label"] == label_val]
            if len(subset):
                print(f"\n-- Sample rows: {LABEL_MAP[label_val]} --")
                sample_cols = [
                    c for c in ["title", "description", "act", "section", "case_type", "label_reason"]
                    if c in df.columns
                ]
                sample = subset.sample(min(SAMPLE_PER_LABEL, len(subset)), random_state=1)
                for _, row in sample.iterrows():
                    for col in sample_cols:
                        val = str(row.get(col, ""))[:120]
                        if val and val != "nan":
                            print(f"  {col}: {val}")
                    print("  ---")

In [5]:
def main():
    print("=" * 60)
    print("Dataset Explorer & Validator")
    print("=" * 60)

    ddl_processed = first_existing([
        COMPILED_DIR_PRIMARY / "ddl_processed.parquet",
        COMPILED_DIR_FALLBACK / "ddl_processed.parquet",
    ])
    if ddl_processed is None:
        ddl_processed = first_matching(
            [COMPILED_DIR_PRIMARY, COMPILED_DIR_FALLBACK],
            "ddl_processed_*_part_*.parquet"
        )

    files = {
        "DDL Processed": ddl_processed,
        "High Court Metadata": first_existing([
            COMPILED_DIR_PRIMARY / "hc_metadata.parquet",
            COMPILED_DIR_FALLBACK / "hc_metadata.parquet",
            IHCJ_DIR / "hc_metadata.parquet",
        ]),
        "Supreme Court Metadata": first_existing([
            COMPILED_DIR_PRIMARY / "sc_metadata.parquet",
            COMPILED_DIR_FALLBACK / "sc_metadata.parquet",
            ISCJ_DIR / "sc_metadata.parquet",
        ]),
        "DDL Labeled": first_existing([
            COMPILED_DIR_PRIMARY / "ddl_labeled.parquet",
            COMPILED_DIR_FALLBACK / "ddl_labeled.parquet",
        ]),
        "High Court Labeled": first_existing([
            COMPILED_DIR_PRIMARY / "hc_labeled.parquet",
            COMPILED_DIR_FALLBACK / "hc_labeled.parquet",
        ]),
        "Supreme Court Labeled": first_existing([
            COMPILED_DIR_PRIMARY / "sc_labeled.parquet",
            COMPILED_DIR_FALLBACK / "sc_labeled.parquet",
        ]),
        "Training Data": first_existing([
            COMPILED_DIR_PRIMARY / "training_data.parquet",
            COMPILED_DIR_FALLBACK / "training_data.parquet",
        ]),
        "Needs LLM Labeling": first_existing([
            COMPILED_DIR_PRIMARY / "needs_llm_labeling.parquet",
            COMPILED_DIR_FALLBACK / "needs_llm_labeling.parquet",
        ]),
        "LLM Labeled Sample": first_existing([
            COMPILED_DIR_PRIMARY / "llm_labeled_sample.parquet",
            COMPILED_DIR_FALLBACK / "llm_labeled_sample.parquet",
        ]),
    }

    for name, path in files.items():
        if path is None:
            print(f"  [SKIP] {name} not found in configured directories.")
            continue
        explore(path, name)

    training_path = files["Training Data"]
    if training_path is not None and training_path.exists():
        show_section("TRAINING DATA QUALITY REPORT")

        needed_cols = ["title", "description", "act", "final_label"]
        df, total_rows, was_truncated = read_parquet_head(
            training_path,
            max_rows=MAX_ROWS_PER_FILE,
            columns=needed_cols,
        )

        if df.empty:
            print("  [SKIP] training_data.parquet is empty or required columns are missing.")
            print("\n\nAll done. Next step: run 07_train_model.py")
            return

        if was_truncated:
            if total_rows is not None:
                print(f"\n  Using sampled rows: {len(df):,} of {total_rows:,}")
            else:
                print(f"\n  Using sampled rows: {len(df):,}")

        has_title = df["title"].notna().sum() if "title" in df.columns else 0
        has_desc = df["description"].notna().sum() if "description" in df.columns else 0
        has_act = df["act"].notna().sum() if "act" in df.columns else 0

        print(f"\n  Rows analyzed:             {len(df):,}")
        print(f"  Rows with title text:      {has_title:,}  ({has_title/len(df)*100:.1f}%)")
        print(f"  Rows with description:     {has_desc:,}  ({has_desc/len(df)*100:.1f}%)")
        print(f"  Rows with act name:        {has_act:,}  ({has_act/len(df)*100:.1f}%)")

        if "final_label" in df.columns and df["final_label"].notna().any():
            counts = df["final_label"].value_counts()
            minority = counts.min()
            majority = counts.max()
            imbalance = majority / minority if minority > 0 else float("inf")
            print(f"\n  Class imbalance ratio:     {imbalance:.1f}x")
            if imbalance > 10:
                print("  High imbalance: consider oversampling minority class (SMOTE)")
            else:
                print("  Imbalance is manageable")

        print("\n  Recommendation:")
        if has_desc > 10000:
            print("  You have enough text data for a BERT/LegalBERT classifier")
        elif has_act > 50000:
            print("  Use structured features (act/section/type) for a baseline RF/XGB model")
        print("  Combine both for best performance")

    print("\n\nAll done. Next step: run 07_train_model.py")

In [6]:
if __name__ == "__main__":
    main()

Dataset Explorer & Validator

════════════════════════════════════════════════════════════
  DDL Processed  (75,000 rows (sampled first 75,000 of 10,475,876 rows))
════════════════════════════════════════════════════════════

Columns (9):
  ddl_case_id, year, state_code, dist_code, type_name_s, disp_name_s, state_name, date_of_filing, date_of_decision

Data types:
ddl_case_id                 object
year                         int16
state_code                    int8
dist_code                     int8
type_name_s               category
disp_name_s               category
state_name                category
date_of_filing      datetime64[ns]
date_of_decision    datetime64[ns]

Null counts (top columns):
date_of_decision    14784

Year range: 2015 - 2015
year
2015    75000

════════════════════════════════════════════════════════════
  High Court Metadata  (75,000 rows (sampled first 75,000 of 3,433,684 rows))
════════════════════════════════════════════════════════════

Columns (15):
  co